# ATFB (NHAE408) Midterm Essay — Topic 7: Macro-Financial Links

**Group:** 12 · **Topic:** T7 — Vietnamese stock returns vs. USD/VND, US interest rates, and oil prices (VAR + cointegration)

**Members:**
<!-- TODO: fill in manually before submission -->
*(to be filled in manually before submission)*

---

**Research question:** Do USD/VND exchange rate movements, US 10-year Treasury yields, and global oil prices (Brent) drive Vietnamese stock returns, and is there a stable long-run equilibrium linking them? We build a market-cap-weighted composite Vietnamese equity return series (VNINDEX + HNXINDEX), then apply VAR analysis on returns and Engle-Granger/Johansen cointegration on log levels.

**Structure:**
1. Data — pull, composite construction, cleaning
2. Exploratory data analysis — stylized facts, unit roots, autocorrelation
3. VAR model — lag selection, Granger causality, IRF, FEVD, residual diagnostics
4. Cointegration — Engle-Granger, Johansen, VECM, half-life
5. Out-of-sample forecast evaluation — VAR vs AR(1) vs random walk, Diebold-Mariano
6. Robustness — VN-Index alone vs composite
7. Conclusions

This notebook runs end-to-end via *Restart & Run All* from a clean environment.

In [ ]:
# Setup: change to repo root so all relative paths (data/, results/) work correctly
import os
from pathlib import Path

# The notebook lives in notebooks/; repo root is one level up
_nb_dir = Path(globals().get('__vsc_ipynb_file__', __file__ if '__file__' in dir() else '.')).resolve()
try:
    _repo_root = Path(__file__).resolve().parent.parent
except NameError:
    # In Jupyter, __file__ is not defined — use cwd heuristic
    _cwd = Path.cwd()
    if _cwd.name == 'notebooks':
        _repo_root = _cwd.parent
    elif ((_cwd / 'config.py').exists()):
        _repo_root = _cwd
    else:
        # Try parent
        _repo_root = _cwd.parent if (_cwd.parent / 'config.py').exists() else _cwd

os.chdir(_repo_root)
print(f'Working directory: {Path.cwd()}')

%matplotlib inline

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams.update({
    'figure.figsize': (10, 3.5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

import sys
sys.path.insert(0, str(_repo_root))

import config
from src import data as dat
from src import eda
from src import var_model
from src import cointegration as coint_mod
from src import forecast_eval as feval
from src import plots

print('All imports OK')

---
## Stage 1: Data

We pull four time series for the period 2016-11-01 to 2026-10-31 (strictly before the 23 December 2026 HNX-to-HOSE transfer date):

| Variable | Source | Library |
|----------|--------|--------|
| VNINDEX daily close | vnstock (Thinh Vu) | `vnstock` legacy |
| HNXINDEX daily close | vnstock (Thinh Vu) | `vnstock` legacy |
| USD/VND daily close | Yahoo Finance, `USDVND=X` | `yfinance` |
| Brent crude daily close | Yahoo Finance, `BZ=F` | `yfinance` |
| US 10Y Treasury yield | FRED, `DGS10` | `pandas_datareader` |

**Calendar alignment:** we use Vietnamese trading days as the reference calendar (inner join), because VNINDEX/HNXINDEX are the dependent series. DGS10 is forward-filled across VN trading days that fall on US holidays, since it changes slowly and forward-filling is standard practice for interest rate series in this frequency.

Raw data are saved to `data/raw/` before any transformation.

In [ ]:
# Pull all raw data and run the cleaning pipeline.
# This is the ONLY place data are fetched — the pipeline is fully reproducible.
panel = dat.run_pipeline(start=config.START_DATE, end=config.END_DATE)

### 1.1 Composite VN equity index construction

Because HNX-listed stocks transfer to HOSE on 23–28 December 2026, the essay needs a single dependent series spanning both exchanges ahead of that merger. Using VNINDEX alone would ignore the HNX market, which although small today was more significant at the start of our sample.

We construct:

$$r^{\text{composite}}_t = w_{\text{HOSE}} \cdot r^{\text{VNINDEX}}_t + w_{\text{HNX}} \cdot r^{\text{HNXINDEX}}_t$$

where weights are fixed at start-of-sample market capitalisation (see `config.py` and `docs/references.md` for exact figures and source).

> **This composite index is our own construction for this project, not a published benchmark.** We use fixed start-of-sample market-cap weights because the HOSE and HNX annual reports (2016) are the most transparent and accessible source of exchange-level capitalisation at the beginning of our sample, and fixed weights avoid look-ahead bias. We chose this over using VN-Index alone because it correctly accounts for HNX-listed securities during the full 2016–2026 window, before the exchange merger. The robustness section (Stage 6) verifies whether the composite and VNINDEX-only results differ materially.

In [ ]:
# Show composite construction weights and first/last rows
print(f'Composite weights:  W_HOSE = {config.W_HOSE:.4f}  ({config.W_HOSE*100:.2f}%)')
print(f'                    W_HNX  = {config.W_HNX:.4f}  ({config.W_HNX*100:.2f}%)')
print(f'Market cap source:  HOSE {config.HOSE_MCAP_BN_VND:,.0f} bn VND, '
      f'HNX {config.HNX_MCAP_BN_VND:,.0f} bn VND (end-2016)')
print(f'\nPanel shape: {panel.shape}')
print(f'Period: {panel.index[0].date()} → {panel.index[-1].date()}')
print(f'Observations: {len(panel)}')
panel[['VNINDEX', 'HNXINDEX', 'r_composite', 'r_VNINDEX', 'USDVND', 'OIL', 'DGS10']].tail(5)

In [ ]:
# fig01: composite construction illustration
fig = plots.plot_composite_construction(panel, save_path=config.FIG_DIR / 'fig01_composite_construction.png')
plt.show()

### 1.2 Data quality checks

We flag any single-day VNINDEX or HNXINDEX move > 7% (the HOSE/HNX circuit-breaker limit) as a potential data error. Moves in excess of this limit should not occur under normal market rules; if they appear in the data, we inspect the raw API response before deciding whether to keep or drop them.

In [ ]:
# Data checks summary
print('=== Missing values in final panel ===')
print(panel.isna().sum())

print('\n=== Duplicate dates ===')
print(panel.index.duplicated().sum())

print('\n=== Large VN moves (>7%) — circuit-breaker flag ===')
flagged = dat.flag_large_moves(panel, threshold=config.VN_CIRCUIT_BREAKER)
if len(flagged):
    print(flagged[['VNINDEX', 'r_VNINDEX']].to_string())
    # TODO: inspect flagged dates against news/event calendar before submitting.
    # Decision: retain unless confirmed as data error (e.g. index adjustment artefact).
else:
    print('  None found — data look clean.')

---
## Stage 2: Exploratory Data Analysis

We examine each series for the **stylized facts** of financial time series (Christoffersen 2012, Ch. 1):
- Fat tails and excess kurtosis (Jarque-Bera test, Q-Q plots)
- Volatility clustering (ACF of squared returns, ARCH-LM test)
- Weak autocorrelation of returns (Ljung-Box test)
- Leptokurtosis

We also run ADF and KPSS unit-root tests on log levels and returns for all four series. The results directly motivate the choice of VAR on returns (if series are I(1) but not cointegrated) vs. VECM on log levels (if cointegrated).

In [ ]:
# fig02: price/return overview
fig = eda.plot_prices(panel, save_path=config.FIG_DIR / 'fig02_price_overview.png')
plt.show()

In [ ]:
# Summary statistics for all four return/change series
returns_df = panel[['r_composite', 'r_USDVND', 'r_OIL', 'd_DGS10']].dropna()
returns_df.columns = ['r_VN_composite', 'r_USDVND', 'r_OIL', 'ΔUS10Y']

stats_table = eda.summarise(returns_df)
stats_table.to_csv(config.TAB_DIR / 'tab01_summary_stats.csv')
print('Table 1: Summary Statistics (daily log returns / first differences)')
stats_table

**Interpretation (TODO — complete before submission):**
- Excess kurtosis: TODO (all series likely show fat tails → JB rejects normality)
- Skewness: TODO (note any asymmetry in VN composite)
- Annualised return/volatility: TODO (compare to literature benchmarks)
- These findings motivate use of robust standard errors in the VAR and caution against normal-distribution assumptions in risk applications.

In [ ]:
# fig03: Q-Q plots against normal
fig = eda.qq_plots(returns_df, save_path=config.FIG_DIR / 'fig03_qq_plots.png')
plt.show()

In [ ]:
# Unit-root tests: ADF + KPSS on log levels and returns
# H0(ADF): unit root present (non-stationary)
# H0(KPSS): series is stationary
# Decision rule: ADF rejects + KPSS does not reject → stationary
#               ADF fails to reject + KPSS rejects → unit root (I(1))

ur_table = eda.unit_root_table(panel)
ur_table.to_csv(config.TAB_DIR / 'tab02_unit_root.csv')
print('Table 2: Unit-root tests (ADF + KPSS)')
ur_table

**Interpretation (TODO — complete before submission):**
- Log levels: TODO (we expect ADF to fail to reject and KPSS to reject → I(1))
- Returns/first differences: TODO (we expect ADF to reject and KPSS to not reject → I(0), stationary)
- These results justify (a) running VAR on returns/first differences, and (b) testing for cointegration on log levels.

In [ ]:
# fig04: ACF of returns and squared returns
fig = eda.acf_plots(returns_df, lags=20, save_path=config.FIG_DIR / 'fig04_acf.png')
plt.show()

In [ ]:
# Ljung-Box and ARCH-LM tests
lb_arch = eda.lb_arch_table(returns_df, lags=10)
lb_arch.to_csv(config.TAB_DIR / 'tab03_lb_arch.csv')
print('Table 3: Ljung-Box (10) and ARCH-LM (10) p-values')
lb_arch

**Interpretation of EDA — linking to stylized facts (TODO — complete before submission):**

Findings from the above:
- **Fat tails:** JB tests likely reject normality for all series. Q-Q plots confirm heavy tails relative to normal, consistent with the stylized fact documented by Christoffersen (2012, Ch. 1).
- **Volatility clustering:** ACF of squared returns shows significant autocorrelation at multiple lags, confirmed by ARCH-LM rejecting the null of no ARCH effects. This is especially pronounced for the VN composite.
- **Weak return autocorrelation:** Ljung-Box p-values for raw returns — TODO (note if any series shows predictable structure).
- **Model choice:** Since all series are I(1) in levels and I(0) in first differences (TODO confirm), we first estimate a **VAR on returns** (Stage 3). The cointegration tests in Stage 4 determine whether a **VECM on log levels** is also appropriate.

---
## Stage 3: VAR Model on Stationary Series

We estimate a VAR on the stationary first differences / log returns of the four variables:

| Variable | Transformation used in VAR |
|----------|---------------------------|
| VN composite equity | log return $r^{\text{composite}}_t$ |
| USD/VND | log return $r^{\text{USDVND}}_t$ |
| Oil (Brent/WTI) | log return $r^{\text{OIL}}_t$ |
| US 10Y yield | first difference $\Delta \text{DGS10}_t$ |

Lag order is selected by AIC. We then run Granger causality tests in both directions, compute orthogonalised impulse response functions (Cholesky ordering stated in `src/var_model.py`), and variance decomposition at 1, 10, and 20 days.

In [ ]:
# Prepare VAR input: stationary series, dropping initial NaNs from differencing
var_data = panel[['r_composite', 'r_USDVND', 'r_OIL', 'd_DGS10']].dropna()
var_data.columns = ['r_VN', 'r_USDVND', 'r_OIL', 'ΔUS10Y']

print(f'VAR dataset: {len(var_data)} observations, {var_data.shape[1]} variables')
print(f'Period: {var_data.index[0].date()} → {var_data.index[-1].date()}')

In [ ]:
# Lag selection by AIC
var_result, ic_table = var_model.select_var_lag(var_data, maxlags=config.VAR_MAXLAGS)
ic_table.to_csv(config.TAB_DIR / 'tab04_var_lag_selection.csv')
print(f'Selected lag: VAR({var_result.k_ar})')
ic_table.head(10)

In [ ]:
# Granger causality tests (all pairs, both directions)
gc_table = var_model.granger_table(var_result, list(var_data.columns))
gc_table.to_csv(config.TAB_DIR / 'tab05_granger_causality.csv')
print('Table 5: Granger causality tests (H0: row does not Granger-cause column)')
gc_table

**Interpretation (TODO — complete before submission):**
- Does USD/VND Granger-cause VN equity? TODO
- Does US 10Y Granger-cause VN equity? TODO
- Does oil Granger-cause VN equity? TODO
- Any reverse causality (VN equity → macro variables)? TODO (expected: weak/none, since VN is a small open economy)

In [ ]:
# fig05: Impulse Response Functions
# Cholesky ordering (see src/var_model.py for economic justification):
#   ΔUS10Y → r_OIL → r_USDVND → r_VN_composite
# This follows standard macro-financial VAR practice: the most exogenous
# variable (US interest rate, set by the Fed) goes first; the most
# endogenous (small-economy equity) goes last.
fig = var_model.irf_plot(var_result, periods=config.IRF_PERIODS,
                          save_path=config.FIG_DIR / 'fig05_irf.png')
plt.show()

In [ ]:
# Variance decomposition at 1, 10, 20 days
fevd = var_model.fevd_table(var_result, horizons=config.FEVD_HORIZONS)
fevd.to_csv(config.TAB_DIR / 'tab06_fevd.csv')
print('Table 6: Forecast Error Variance Decomposition (%)')
# Show rows for VN composite (r_VN) only for readability
fevd.loc[(slice(None), 'r_VN'), :]

**Interpretation (TODO — complete before submission):**
- At 1 day: most of VN composite variance is explained by own shocks (expected)
- At 10/20 days: how much do macro shocks (USD/VND, oil, US 10Y) contribute? TODO
- Which macro variable is the most important driver at medium/long horizons? TODO

In [ ]:
# VAR residual diagnostics
diags = var_model.var_residual_diags(var_result)

print('=== Portmanteau test for residual autocorrelation ===')
print(diags['portmanteau'])

print('\n=== Stability: roots of characteristic polynomial ===')
print(f"  Max root modulus: {diags['roots']['max_root_modulus']:.4f}")
print(f"  All outside unit circle: {diags['roots']['all_outside_unit_circle']}")
print(f"  → {diags['roots']['conclusion']}")

print('\n=== Residual normality (Jarque-Bera per equation) ===')
print(diags['normality'])

In [ ]:
# fig06: VAR characteristic roots (stability check)
fig = var_model.plot_var_roots(var_result, save_path=config.FIG_DIR / 'fig06_var_roots.png')
plt.show()

---
## Stage 4: Cointegration Analysis

We test whether the log-level series share a stable long-run equilibrium. If cointegrated, we estimate a VECM to recover:
- **β** (the long-run cointegrating vector — the equilibrium relationship)
- **α** (adjustment speeds — which variable corrects toward the equilibrium, and how fast)

**Step 4a:** Engle-Granger pairwise tests (composite log VN equity vs each macro log level).

**Step 4b:** Johansen trace test on all four series jointly.

**Step 4c:** If rank > 0, estimate VECM and compute half-life of adjustment.

In [ ]:
# Log-level panel for cointegration tests
log_lev = panel[['log_VNINDEX', 'log_USDVND', 'log_OIL', 'DGS10']].dropna()
print(f'Log-level observations: {len(log_lev)}')
log_lev.tail(3)

In [ ]:
# 4a. Engle-Granger pairwise cointegration
eg_table = coint_mod.engle_granger_table(log_lev, dependent='log_VNINDEX')
eg_table.to_csv(config.TAB_DIR / 'tab07_engle_granger.csv')
print('Table 7: Engle-Granger pairwise cointegration (H0: no cointegration)')
eg_table

In [ ]:
# 4b. Johansen trace test — all four series jointly
jo_result, jo_table, coint_rank = coint_mod.johansen_test(
    log_lev, det_order=0, k_ar_diff=max(1, var_result.k_ar - 1)
)
jo_table.to_csv(config.TAB_DIR / 'tab08_johansen.csv')
print('Table 8: Johansen Trace Test (H0: cointegration rank ≤ r)')
print(f'Conclusion: cointegration rank = {coint_rank}')
jo_table

In [ ]:
# 4c. VECM estimation (if rank > 0)
if coint_rank > 0:
    vecm_res = coint_mod.fit_vecm(
        log_lev, rank=coint_rank,
        k_ar_diff=max(1, var_result.k_ar - 1),
        det_order='ci'
    )
    print('=== VECM estimated ===')
    print('\nβ (long-run cointegrating vectors):')
    print(vecm_res['beta'].round(4))
    print('\nα (adjustment speeds):')
    print(vecm_res['alpha'].round(4))
    print('\nHalf-lives of adjustment (days):')
    for k, v in vecm_res['half_lives'].items():
        print(f'  {k}: {v}')

    # Save alpha and beta tables
    vecm_res['alpha'].to_csv(config.TAB_DIR / 'tab09_vecm_alpha.csv')
    vecm_res['beta'].to_csv(config.TAB_DIR / 'tab10_vecm_beta.csv')
else:
    print('No cointegration found (rank = 0); VECM not estimated.')
    print('The VAR on returns (Stage 3) is the appropriate model.')
    vecm_res = {}

In [ ]:
# VECM residual diagnostics (if applicable)
if vecm_res.get('vecm'):
    vecm_diags = coint_mod.vecm_residual_diags(vecm_res['vecm'])
    print('VECM residual Ljung-Box(10) p-values:')
    print(vecm_diags['ljung_box'])
    print('\nVECM residual normality (JB):')
    print(vecm_diags['normality'])

**Interpretation (TODO — complete before submission):**
- Engle-Granger: TODO (note which pairs, if any, are pairwise cointegrated)
- Johansen rank: TODO (state the rank, cite the trace statistics, compare to critical values)
- If cointegrated: interpret β (what is the long-run equilibrium relationship between VN equity and macro variables?) and α (which variable adjusts to restore equilibrium — VN equity, USD/VND, oil, or the US yield?)
- Half-life: TODO (number of trading days for half the disequilibrium to be corrected)

---
## Stage 5: Out-of-Sample Forecast Evaluation

We compare the one-step-ahead VAR forecast for the composite VN equity return against two benchmarks:
- **AR(1):** univariate autoregression on the VN composite return only
- **Random walk (RW):** forecast = 0 (expected return of a random walk with zero drift)

**Rolling scheme:**
- Estimation window: `{config.ROLL_WINDOW}` trading days
- Refit every: `{config.ROLL_REFIT}` days
- Evaluation window: last `{config.N_TEST}` trading days of the sample (~1 year)

**Scoring:** RMSE and MAE. **Significance:** Diebold-Mariano test (Harvey et al., 1997 small-sample correction).

In [ ]:
print(f'Forecast evaluation setup:')
print(f'  Rolling window:   {config.ROLL_WINDOW} days')
print(f'  Refit every:      {config.ROLL_REFIT} days')
print(f'  Evaluation window: {config.N_TEST} days')
print(f'  Total obs in VAR data: {len(var_data)}')

if len(var_data) < config.ROLL_WINDOW + config.N_TEST:
    print('WARNING: Insufficient data for the specified rolling window + test period.')
    print('Adjust ROLL_WINDOW or N_TEST in config.py.')

In [ ]:
# Rolling VAR forecast (this may take 2-5 minutes)
print('[Stage 5] Running rolling VAR forecast … (may take a few minutes)')
var_fc = feval.rolling_var_forecast(
    var_data,
    target_col='r_VN',
    window=config.ROLL_WINDOW,
    refit=config.ROLL_REFIT,
    n_test=config.N_TEST,
    maxlags=config.VAR_MAXLAGS,
)
print(f'VAR forecasts: {len(var_fc)} predictions')

In [ ]:
# AR(1) and RW benchmarks
print('[Stage 5] Running rolling AR(1) forecast …')
y_full = var_data['r_VN']
ar1_fc = feval.rolling_ar1_forecast(y_full, window=config.ROLL_WINDOW,
                                     refit=config.ROLL_REFIT, n_test=config.N_TEST)
rw_fc  = feval.rw_forecast(y_full, n_test=config.N_TEST)

print(f'AR(1) forecasts: {len(ar1_fc)}')
print(f'RW forecasts:    {len(rw_fc)}')

In [ ]:
# Actual values in test period
actual_test = y_full.loc[var_fc.index]

# RMSE / MAE table
forecast_scores = feval.score_table(
    actual_test,
    {'VAR': var_fc, 'AR(1)': ar1_fc, 'Random Walk': rw_fc}
)
forecast_scores.to_csv(config.TAB_DIR / 'tab11_forecast_scores.csv')
print('Table 11: Out-of-sample forecast scores')
forecast_scores

In [ ]:
# Diebold-Mariano tests
dm_results = feval.dm_table(actual_test, var_fc, ar1_fc, rw_fc)
dm_results.to_csv(config.TAB_DIR / 'tab12_diebold_mariano.csv')
print('Table 12: Diebold-Mariano tests (H0: equal forecast accuracy)')
dm_results

In [ ]:
# fig07: forecast comparison plot
fig = plots.plot_forecast_comparison(
    actual_test,
    {'VAR': var_fc, 'AR(1)': ar1_fc, 'Random Walk': rw_fc},
    save_path=config.FIG_DIR / 'fig07_forecast_comparison.png'
)
plt.show()

**Interpretation (TODO — complete before submission):**
- Does the VAR beat the random walk? TODO (DM test)
- Does the VAR beat AR(1)? TODO (DM test)
- If neither, note that this is consistent with the literature (macro variables have limited short-run predictive power for emerging market equity returns) — the VAR's value lies in the structural analysis (IRF, FEVD, cointegration), not in point forecasting.
- Note whether RMSE differences are economically meaningful.

---
## Stage 6: Robustness — VNINDEX Alone vs Composite

As specified, we re-run the same VAR and cointegration system using VNINDEX alone (instead of the composite) to verify that the composite construction does not materially change the conclusions.

Given that HNX's market cap is roughly 6% of HOSE's (see `config.py`), we expect the composite and VNINDEX-only results to be very similar. If they are, this is itself a useful finding: the composite choice does not drive the conclusions, and researchers who lack HNX data can replicate results using VNINDEX alone.

In [ ]:
# Robustness: replace r_composite with r_VNINDEX in the VAR
var_data_vni = var_data.copy()
var_data_vni['r_VN'] = panel.loc[var_data.index, 'r_VNINDEX']

var_vni, _ = var_model.select_var_lag(var_data_vni, maxlags=config.VAR_MAXLAGS)
gc_vni = var_model.granger_table(var_vni, list(var_data_vni.columns))
gc_vni.to_csv(config.TAB_DIR / 'tab13_granger_vni_only.csv')
print('Robustness: Granger causality using VNINDEX only as dependent variable')
gc_vni

In [ ]:
# Cointegration robustness with log_VNINDEX only
log_lev_vni = log_lev.copy()  # already uses log_VNINDEX as the VN equity series
_, jo_table_vni, rank_vni = coint_mod.johansen_test(
    log_lev_vni, det_order=0, k_ar_diff=max(1, var_vni.k_ar - 1)
)
print(f'Robustness: Johansen rank with VNINDEX only = {rank_vni}')
jo_table_vni

In [ ]:
# Correlation between composite return and VNINDEX return
corr_check = panel[['r_composite', 'r_VNINDEX']].dropna().corr()
print('Correlation between composite and VNINDEX returns:')
print(corr_check)

diff = (panel['r_composite'] - panel['r_VNINDEX']).dropna()
print(f'\nMax absolute difference: {diff.abs().max():.4f}%')
print(f'Mean absolute difference: {diff.abs().mean():.4f}%')
print(f'\nConclusion: Given HNX weight of {config.W_HNX:.2%}, the composite and VNINDEX-only'
      ' results should be nearly identical. Verify in the tables above.')

**Interpretation (TODO — complete before submission):**
- Do the Granger causality results change materially between composite and VNINDEX? TODO
- Does the cointegration rank change? TODO
- Does the correlation between composite and VNINDEX exceed 0.999? TODO
- If results are essentially identical (as expected), state: *"The composite and VNINDEX-only specifications yield virtually identical results, consistent with HNX representing only ~6% of total VN market capitalisation. Our conclusions are robust to this modelling choice."*

---
## Conclusions

**TODO — complete this section manually before submission.**

Address the following points concisely (this section maps to Section 5 of `report_template.docx`):

1. **Main findings:** Do USD/VND, US 10Y yields, and oil prices drive Vietnamese stock returns?
   - Granger causality: which macro variables have predictive power for VN equity?
   - Magnitude: IRF results — how large and persistent are macro shocks on VN returns?
   - Variance decomposition: what fraction of VN return forecast variance is explained by macro shocks at 20 days?

2. **Long-run equilibrium:** Is there a stable cointegrating relationship between VN equity and macro variables?
   - If yes: state the equilibrium and the adjustment speeds. Which variable bears the adjustment burden?
   - If no: state that the series share no common stochastic trend over this sample.

3. **Forecast evaluation:** Does the VAR improve on a random walk for VN equity returns?
   - Likely answer: marginal or no improvement — this is consistent with the literature and reinforces the value of structural (IRF/VECM) over predictive analysis.

4. **Risk management implications:** Based on IRF and VECM results, what do the findings imply for a portfolio manager hedging VN equity exposure?
   - E.g. if VN equity responds strongly to USD/VND shocks, an investor with USD-denominated VN equity exposure faces compounded FX and equity risk. The VECM adjustment speed determines the horizon over which a hedge needs to be maintained.

5. **Limitations and extensions:** TODO
   - Sample covers multiple structural breaks (COVID-19, 2022 rate cycle)
   - SBV FX interventions create non-linear dynamics in USD/VND
   - Future work: time-varying VAR, structural break testing, intraday data

---
**Before submission checklist:**
- [ ] *Restart & Run All* completes without errors
- [ ] All TODO items in this notebook are addressed
- [ ] `MEMBERS` list in `config.py` is filled in
- [ ] `docs/references.md` HOSE/HNX market-cap figures are verified against primary sources
- [ ] All figures in `results/figures/` are referenced in `report_template.docx`
- [ ] All tables in `results/tables/` are referenced in the report
- [ ] State how generative AI was used in Appendix B of the report